# Training a CNN on MNIST with PyTorch Lightning

This notebook demonstrates how to train a simple, efficient Convolutional Neural Network (CNN) with residual connections on the MNIST dataset using PyTorch Lightning. The configuration is tuned for ≥99.7% accuracy.

In [1]:
!pip install pytorch-lightning torch torchvision datasets wandb pillow scikit-learn


[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
from PIL import ExifTags, Image
Image.ExifTags = ExifTags  # Hack to bypass broken import

In [3]:
# --- Config Setup ---
import os

def notebook_id_from_title():
    # Dummy implementation for notebook ID
    return "mnist-cnn-pl"

def setup_config():
    # General Settings
    dataset = "cifar10"  # Options: "mnist" or "cifar10"
    seed = 42
    n_epochs = 100
    batch_size = 2056
    # Optimizer Settings
    learning_rate = 1e-3
    weight_decay = 0
    warmup_ratio = 0
    max_grad_norm = 0
    # Model Architecture
    hidden_size = 128
    nonlinearity = 'relu'
    weight_init = 'kaiming'
    # Data Settings
    sequence_length = 1
    train_size = 0.9
    val_size = 0.1
    input_size = 1
    output_size = 10
    # Softcoded new params
    label_smoothing = 0.1
    dropout = 0.1
    early_stopping_patience = 7
    early_stopping_min_delta = 1e-4
    use_mixed_precision = True
    swa_lrs = 1e-2
    activation = "silu"

    #os.environ["NOTEBOOK_ID"] = notebook_id_from_title()

    return {
        'dataset': dataset,
        'seed': seed,
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'sequence_length': sequence_length,
        'input_size': input_size,
        'hidden_size': hidden_size,
        'output_size': output_size,
        'nonlinearity': nonlinearity,
        'weight_init': weight_init,
        'weight_decay': weight_decay,
        'warmup_ratio': warmup_ratio,
        'max_grad_norm': max_grad_norm,
        'train_size': train_size,
        'val_size': val_size,
        'label_smoothing': label_smoothing,
        'dropout': dropout,
        'early_stopping_patience': early_stopping_patience,
        'early_stopping_min_delta': early_stopping_min_delta,
        'use_mixed_precision': use_mixed_precision,
        'swa_lrs': swa_lrs,
        'activation': activation
    }

CONFIG = setup_config()

In [4]:
import pytorch_lightning as pl
pl.seed_everything(CONFIG['seed'], workers=True)

Seed set to 42


42

In [5]:
import torch
torch.set_float32_matmul_precision('high')  # or 'medium'

In [6]:
import wandb
wandb.login()

wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
from torch import nn
from torchvision.datasets import MNIST, CIFAR10

def get_activation():
    from torch import nn
    act = CONFIG.get('activation', 'silu')
    if act.lower() == "silu": return nn.SiLU()
    elif act.lower() == "relu": return nn.ReLU()
    elif act.lower() == "leaky_relu": return nn.LeakyReLU()
    elif act.lower() == "tanh": return nn.Tanh()
    elif act.lower() == "sigmoid": return nn.Sigmoid()
    else: raise ValueError(f"Unsupported activation: {act}")
    
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=None):
        super().__init__()

        dropout = CONFIG['dropout']

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = get_activation()
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if dropout: self.dropout = nn.Dropout2d(dropout)
        self.downsample = None

        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, 0, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        dropout = CONFIG['dropout']

        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if dropout: out = self.dropout(out)
        if self.downsample is not None: identity = self.downsample(x)
        out += identity
        out = get_activation()(out)
        return out


In [8]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms

class FlexibleDataModule(pl.LightningDataModule):
    def __init__(self, dataset="mnist", batch_size=None, num_workers=2):
        super().__init__()
        self.dataset = dataset.lower()
        self.batch_size = batch_size if batch_size is not None else CONFIG['batch_size']
        self.num_workers = num_workers

        if self.dataset == "mnist":
            self.input_channels = 1
            self.n_classes = 10
            self.train_transform = transforms.Compose([
                transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
                transforms.RandomRotation(degrees=15),
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
                transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value='random')
            ])
            self.test_transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,))
            ])
        elif self.dataset == "cifar10":
            self.input_channels = 3
            self.n_classes = 10
            self.train_transform = transforms.Compose([
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
            ])
            self.test_transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
            ])
        else:
            raise ValueError("Unsupported dataset: choose 'mnist' or 'cifar10'")

    def prepare_data(self):
        if self.dataset == "mnist":
            MNIST(root="./data", train=True, download=True)
            MNIST(root="./data", train=False, download=True)
        elif self.dataset == "cifar10":
            CIFAR10(root="./data", train=True, download=True)
            CIFAR10(root="./data", train=False, download=True)

    def setup(self, stage=None):
        if self.dataset == "mnist":
            full = MNIST(root="./data", train=True, transform=self.train_transform)
        elif self.dataset == "cifar10":
            full = CIFAR10(root="./data", train=True, transform=self.train_transform)
        total = len(full)
        train_size = int(total * CONFIG['train_size'])
        val_size = total - train_size
        self.train_set, self.val_set = torch.utils.data.random_split(
            full, [train_size, val_size], generator=torch.Generator().manual_seed(CONFIG['seed'])
        )
        self.val_set.dataset.transform = self.test_transform
        if self.dataset == "mnist":
            self.test_set = MNIST(root="./data", train=False, transform=self.test_transform)
        elif self.dataset == "cifar10":
            self.test_set = CIFAR10(root="./data", train=False, transform=self.test_transform)

    def train_dataloader(self):
        return DataLoader(self.train_set, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_set, batch_size=self.batch_size, num_workers=self.num_workers, pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self.test_set, batch_size=self.batch_size, num_workers=self.num_workers, pin_memory=True)

# To use MNIST or CIFAR-10, set CONFIG['dataset'] at the top of the notebook.
dm = FlexibleDataModule(dataset=CONFIG['dataset'], batch_size=CONFIG['batch_size'])

In [9]:
import torch.nn.init as init

class LitCNN(pl.LightningModule):
    def __init__(self, lr=None, input_channels=None, n_classes=None):
        super().__init__()
        self.save_hyperparameters()

        dropout = CONFIG['dropout']
        label_smoothing = CONFIG['label_smoothing']

        # Use input_channels and n_classes for flexibility
        if input_channels is None:
            input_channels = 1  # fallback for MNIST
        if n_classes is None:
            n_classes = 10

        self.stem = nn.Sequential(
            nn.Conv2d(input_channels, 32, 3, 1, 1, bias=False),
            nn.BatchNorm2d(32),
            get_activation()
        )
        self.layer1 = ResidualBlock(32, 64, stride=1, dropout=dropout)
        self.layer2 = ResidualBlock(64, 128, stride=2, dropout=dropout)
        self.layer3 = ResidualBlock(128, 256, stride=2, dropout=dropout)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            get_activation(),
            nn.Dropout(CONFIG['dropout']),
            nn.Linear(128, n_classes)
        )
        self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.validation_step_outputs = []
        self.apply(self._init_weights)

    def _init_weights(self, m):
        # Map config activation to PyTorch nonlinearity string
        nonlinearity = CONFIG['activation'].lower()
        if nonlinearity == "silu": nonlinearity = "relu"

        if isinstance(m, nn.Conv2d):
            init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity=nonlinearity)
            if getattr(m, "bias", None) is not None:
                init.constant_(m.bias, 0)
        elif isinstance(m, nn.Linear):
            init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity=nonlinearity)
            if m.bias is not None:
                init.constant_(m.bias, 0)
        elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            if hasattr(m, "weight") and m.weight is not None:
                init.constant_(m.weight, 1)
            if hasattr(m, "bias") and m.bias is not None:
                init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.global_pool(x)
        x = self.fc(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        self.log("val_loss", loss, prog_bar=True, sync_dist=True)
        self.log("val_acc", acc, prog_bar=True, sync_dist=True)
        return {}

    def on_validation_epoch_end(self):
        self.validation_step_outputs.clear()

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr if hasattr(self.hparams, 'lr') and self.hparams.lr is not None else CONFIG['learning_rate'],
            weight_decay=CONFIG['weight_decay']
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2, verbose=True
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1
            }
        }
    
model = LitCNN(
    lr=CONFIG['learning_rate'],
    input_channels=dm.input_channels if hasattr(dm, "input_channels") else 1,
    n_classes=dm.n_classes if hasattr(dm, "n_classes") else 10
)

In [10]:
import time
from pytorch_lightning.callbacks import Timer
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.callbacks import StochasticWeightAveraging

class EpochTimeLogger(pl.Callback):
    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_start_time = time.time()

    def on_train_epoch_end(self, trainer, pl_module):
        epoch_time = time.time() - self.epoch_start_time
        if trainer.logger is not None and hasattr(trainer.logger, "experiment"):
            trainer.logger.experiment.log({"epoch_time_sec": epoch_time, "epoch": trainer.current_epoch})

trainer_callbacks = [
    Timer(), 
    EpochTimeLogger(), 
    ModelCheckpoint(
        monitor="val_loss",
        save_top_k=1,
        mode="min",
        save_last=True,
        dirpath="checkpoints",
        filename="mnist-cnn-{epoch:02d}-{val_loss:.2f}"
    ), 
    LearningRateMonitor(logging_interval="epoch"),
    EarlyStopping(
        monitor="val_loss",
        patience=CONFIG['early_stopping_patience'],
        min_delta=CONFIG['early_stopping_min_delta'],
        mode="min",
        verbose=True,
        strict=True
    ) if CONFIG['early_stopping_patience'] > 0 else None,
    StochasticWeightAveraging(swa_lrs=CONFIG['swa_lrs']) if CONFIG['swa_lrs'] > 0 else None
]
trainer_callbacks = [cb for cb in trainer_callbacks if cb is not None]

In [11]:
from pytorch_lightning.loggers import WandbLogger

num_gpus = torch.cuda.device_count()

trainer = pl.Trainer(
    devices=num_gpus,
    accelerator="auto",
    strategy="auto",
    benchmark=True,
    max_epochs=CONFIG['n_epochs'],
    logger=WandbLogger(project="mnist-cnn-pl"),
    callbacks=trainer_callbacks,
    precision="16-mixed" if CONFIG['use_mixed_precision'] else 32
)
trainer.fit(model, datamodule=dm)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/3
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/3
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/3
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/3
Initializing distributed: GLOBAL_RANK: 2, MEMBER: 3/3
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 3 processes
----------------------------------------------------------------------------------------------------

Initializing distributed: GLOBAL_RANK: 2, MEMBER: 3/3
----------------------------------------------------------------------------------------------------
distrib

100%|██████████| 170498071/170498071 [00:09<00:00, 17874278.78it/s]


Extracting ./data/cifar-10-python.tar.gz to ./data

Files already downloaded and verified
Files already downloaded and verified


/usr/local/lib/python3.10/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2]
LOCAL_RANK: 2 - CUDA_VISIBLE_DEVICES: [0,1,2]
LOCAL_RANK: 1 - CUDA_VISIBLE_DEVICES: [0,1,2]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2]
LOCAL_RANK: 2 - CUDA_VISIBLE_DEVICES: [0,1,2]
LOCAL_RANK: 1 - CUDA_VISIBLE_DEVICES: [0,1,2]

  | Name        | Type              | Params | Mode 
----------------------------------------------------------
0 | stem        | Sequential        | 928    | train
1 | layer1      | ResidualBlock     | 57.7 K | train
2 | layer2      | ResidualBlock     | 230 K  | train
3 | layer3      | ResidualBlock     | 919 K  | train
4 | global_pool | AdaptiveAvgPool2d | 0      | train
5 | fc          | Sequential        | 34.4 K | train
6 | criterion   | CrossEntropyLoss  | 0      | train
----------------------------------------------------------
1.2 M     Trainable params
0 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 00050: reducing learning rate of group 0 to 3.1250e-05.Epoch 00050: reducing learning rate of group 0 to 3.1250e-05.Epoch 00050: reducing learning rate of group 0 to 3.1250e-05.


Epoch 00050: reducing learning rate of group 0 to 3.1250e-05.Epoch 00050: reducing learning rate of group 0 to 3.1250e-05.




Validation: |          | 0/? [00:00<?, ?it/s]

[rank: 0] Monitored metric val_loss did not improve in the last 7 records. Best score: 1.145. Signaling Trainer to stop.
[rank: 1] Monitored metric val_loss did not improve in the last 7 records. Best score: 1.145. Signaling Trainer to stop.
[rank: 2] Monitored metric val_loss did not improve in the last 7 records. Best score: 1.145. Signaling Trainer to stop.
[rank: 1] Monitored metric val_loss did not improve in the last 7 records. Best score: 1.145. Signaling Trainer to stop.
[rank: 2] Monitored metric val_loss did not improve in the last 7 records. Best score: 1.145. Signaling Trainer to stop.
/usr/local/lib/python3.10/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/us

In [12]:
single_gpu_trainer = pl.Trainer(
    devices=1,
    accelerator="auto"
)
single_gpu_trainer.test(model, datamodule=dm)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/logger_co

Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2]


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.7328000068664551
        test_loss           1.1572046279907227
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 1.1572046279907227, 'test_acc': 0.7328000068664551}]